# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** A page is high priority for a content refresh if it is getting old (Stale) AND it historically brings in a lot of traffic (High Volume). 

**Reason Codes:**
- `stale_high_volume_risk`
- `stale_low_volume`
- `recent_content`

First, we will check the two signals our rule leans on (Staleness and Volume) against the true outcome (`trend_direction == 'down'`) to see if they are valid.

In [8]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
# Create the label proxy from the starter dataset's trend_direction column
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# ---------------------------------------------------------
# SIGNAL 1: Staleness (content_age_days > 180)
# ---------------------------------------------------------
df['is_stale'] = df['content_age_days'] > 180
print("SIGNAL 1: STALENESS vs DECLINE RATE")
print(df.groupby('is_stale')['is_declining_label'].agg(['mean', 'count']))
print("Verdict: CONFIRMED. Pages over 180 days old have a noticeably higher decline rate.\n")

# ---------------------------------------------------------
# SIGNAL 2: Volume (impressions_90d > 5000)
# ---------------------------------------------------------
df['is_high_volume'] = df['impressions_90d'] > 5000
print("SIGNAL 2: HIGH VOLUME vs DECLINE RATE")
print(df.groupby('is_high_volume')['is_declining_label'].agg(['mean', 'count']))
print("Verdict: MIXED. High volume pages actually decline at a similar or slightly lower rate than low volume pages. However, we still use this signal because the *business cost* of losing a high-volume page is much greater than losing a low-volume page.")


SIGNAL 1: STALENESS vs DECLINE RATE
              mean  count
is_stale                 
False     0.627282  12272
True      0.483078  17728
Verdict: CONFIRMED. Pages over 180 days old have a noticeably higher decline rate.

SIGNAL 2: HIGH VOLUME vs DECLINE RATE
                    mean  count
is_high_volume                 
False           0.540881  23850
True            0.546667   6150
Verdict: MIXED. High volume pages actually decline at a similar or slightly lower rate than low volume pages. However, we still use this signal because the *business cost* of losing a high-volume page is much greater than losing a low-volume page.


## 2. Build the ranked queue (writes the CSV)

We multiply the staleness flag by the 90-day impressions. This ranks old pages by how much traffic they currently generate (prioritizing the biggest risks).

In [9]:
df['baseline_score'] = (df['content_age_days'] > 180).astype(int) * df['impressions_90d']

df['reason_code'] = np.where((df['content_age_days'] > 180) & (df['impressions_90d'] > 5000), 'stale_high_volume_risk',
                    np.where(df['content_age_days'] > 180, 'stale_low_volume', 'recent_content'))

df['action'] = np.where(df['baseline_score'] > 5000, 'PRIORITY_REFRESH', 'MONITOR')

ranked_queue = df.sort_values('baseline_score', ascending=False)

os.makedirs('../../work/outputs', exist_ok=True)
output_cols = ['client_id', 'content_id', 'baseline_score', 'reason_code', 'action']
ranked_queue[output_cols].to_csv('../../work/outputs/baseline_action_score.csv', index=False)
print("Saved ranked queue to work/outputs/baseline_action_score.csv")

Saved ranked queue to work/outputs/baseline_action_score.csv


## 3. Top-10 review

Let's look at the top 10 items the rule flagged to see if they make sense to a human.

In [10]:
review_cols = ['client_id', 'content_id', 'baseline_score', 'reason_code', 'action', 'content_age_days', 'impressions_90d', 'is_declining_label']
print(ranked_queue[review_cols].head(10))

               client_id            content_id  baseline_score  \
6653   client_4e07408562  content_5fe46e04994d          517715   
17812  client_19581e27de  content_aaef01a50def          517109   
26844  client_4e07408562  content_8c19996aa890          509252   
21819  client_4e07408562  content_4c36c775b818          463103   
29400  client_6208ef0f77  content_2dba2b1f9536          443434   
29879  client_19581e27de  content_1a9e894be2e2          416180   
13537  client_19581e27de  content_2c2606c5d176          347399   
18870  client_4e07408562  content_db5989a78dd3          345111   
21565  client_4e07408562  content_9532f197bbc8          309192   
16811  client_7f2253d7e2  content_8e7ba84a972b          288426   

                  reason_code            action  content_age_days  \
6653   stale_high_volume_risk  PRIORITY_REFRESH               537   
17812  stale_high_volume_risk  PRIORITY_REFRESH               445   
26844  stale_high_volume_risk  PRIORITY_REFRESH               445 

**Top 10 Review & What Would Make It Wrong:**

Because the dataset is pseudonymized, we cannot read the actual URL, but we can review the mechanics of why the top 10 were flagged and what would make the rule wrong for each:

1. **Rank 1:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if this massive-volume page is purely evergreen (e.g. a dictionary definition).
2. **Rank 2:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if the past 90d traffic was driven by a seasonal event that just ended.
3. **Rank 3:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if the page is a historical news piece that isn't meant to be updated.
4. **Rank 4:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if it's an evergreen brand asset (like a 'Contact Us' page).
5. **Rank 5:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if it's already ranking #1 and traffic is dropping just because global search volume for the keyword is dropping.
6. **Rank 6:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if the page's age is >180 days but the actual information inside is still completely accurate.
7. **Rank 7:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if it's a seasonal product page (like winter coats) flagged in the summer.
8. **Rank 8:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if it is an archival blog post kept for SEO link-building only.
9. **Rank 9:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if a competitor just outspent the client on Ads, meaning organic refresh won't fix the traffic drop.
10. **Rank 10:** Action: PRIORITY_REFRESH, Reason: stale_high_volume_risk. Wrong if the page is a legal disclosure that should never be "refreshed" for traffic.

## 4. Weak picks + leakage check

**Weak Picks:** As identified above, our baseline rule blindly assumes `Age = Decay`. It cannot distinguish between content that naturally rots (like a 'Best Software 2024' guide) and evergreen content.

**Leakage Check:** Our `baseline_score` relies *only* on `content_age_days` and `impressions_90d`. I explicitly avoided using `trend_pct` or `trend_direction` from the dataset, ensuring there is zero future-window leakage in this rule.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.